# R515B Soğutucusu için STGP‑EF Tabanlı Özellik Zenginleştirme ve Regresyon Performans Analizi

Bu çalışma, R515B termodinamik soğutucu akışkanının doymuş ve kızgın fazlarındaki veri setlerinde:
- Girdi kısıtını azaltmak amacıyla **STGP‑EF hibrit yöntemi** ile girdi uzayını zenginleştirmeyi,
- Zenginleştirmenin **11 farklı regresyon algoritmasına** etkisini **5-Fold Çapraz Doğrulama** ile test etmeyi,
- Üretilen hibrit özniteliklerin "Evrensel Permutation Importance" ile ortak önemini belirlemeyi,
- Modellerin termodinamik yasalara uyumunu sentetik ızgara üzerinde test etmeyi amaçlamaktadır.

*Tüm analizler K-Fold üzerinden tek bir döngüde gerçekleştirilecek ve sonuçlar otomatik olarak `Results` klasörüne çok sekmeli Excel formatında kaydedilecektir.*

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

# Functions.py'den gerekli sabitler ve optimize edilmiş ana fonksiyonlar içe aktarılıyor
from Functions2 import (
    SATURATED_INPUTS, 
    SATURATED_OUTPUTS, 
    SUPERHEATED_INPUTS, 
    SUPERHEATED_OUTPUTS,
    run_unified_analysis,
    get_regressors,
    check_thermodynamics_on_synthetic_grid
)

import warnings
warnings.filterwarnings('ignore')

2026-05-22 16:38:07,689 - INFO - Loading faiss with AVX2 support.
2026-05-22 16:38:08,597 - INFO - Successfully loaded faiss with AVX2 support.
c:\Users\SGUZEY\Documents\Yüksek Lisans Dersleri\Veri Mühendisliği\Data_Engineering_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Uyarı: Kendi dizininize ve dosya adınıza göre veri okuma kodunu güncelleyebilirsiniz.
# Örnek okuma kodları:
df_doymus = pd.read_csv("R515B_Ozellikler\R515B_Doymus_Ozellikleri.csv") 
df_kizgin = pd.read_csv("R515B_Ozellikler\R515B_Kizgin_Buhar.csv") 

print("Doymuş Faz Veri Boyutu:", df_doymus.shape)
print("Kızgın Faz Veri Boyutu:", df_kizgin.shape)

Doymuş Faz Veri Boyutu: (136, 8)
Kızgın Faz Veri Boyutu: (578, 5)


In [3]:
# DOYMUŞ FAZ ANALİZİ
# K-Fold, Performans Metrikleri, Öznitelik Katkıları ve Excel Raporlaması tek seferde çalıştırılır.

sat_metrics, sat_importances, sat_final_matrix = run_unified_analysis(
    df=df_doymus,                  
    input_cols=SATURATED_INPUTS, 
    outputs_list=SATURATED_OUTPUTS, 
    dataset_name="Doymus_Faz"
)

print("\n--- Eğitime Giren Nihai Zenginleştirilmiş Veri Matrisi (İlk 5 Satır) ---")
display(sat_final_matrix.head())


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  DOYMUS_FAZ - KESİN SIRALI K-FOLD ANALİZİ VE EXCEL RAPORLAMA
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

[FAZ 1/3] Her hedef değişken için ayrı ayrı hibrit veri setleri üretiliyor...
  -> Hedef (P(çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (v sıvı (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (v buhar (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (h sıvı (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (h buhar (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (s sıvı (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (s buhar (çıktı)) için STGP-EF çalıştırılıyor...

[FAZ 2/3] Orijinal (Base) veri setleri ile modeller eğitiliyor...
      -> AdaBoost eğitiliyor (Base)...
      -> CatBoost eğitiliyor (Base)...
      -> DART eğitiliyor (Base)...
      -> ET eğitiliyor (Base)...
      -> GBDT eğitiliyor (Base)...
      -> KNN eğitiliyor (Base)...
      -> LightGBM eğitiliyor (Base)...
     

,T(girdi),1/T,ln(T),STGP_00,STGP_01,STGP_02,STGP_03,STGP_04,STGP_05,STGP_06,STGP_07,STGP_08,STGP_09,EF_00,EF_01,EF_02,EF_03,EF_04,EF_05,EF_06,EF_07,EF_08,EF_09,Target
0,-30.0000,0.0041,5.4937,0.1623,0.1623,0.1623,0.1623,0.1623,0.1623,0.1623,0.1623,0.1623,0.1623,-1.8532,-2.8532,1.9936,-1.7194,3.8469,-0.8644,-1.8801,-3.5726,-0.8165,-0.8309,61.2210
1,-29.0000,0.0041,5.4978,0.1638,0.1638,0.1638,0.1638,0.1638,0.1638,0.1638,0.1638,0.1638,0.1638,-1.8211,-2.8211,1.9537,-1.6939,3.7748,-0.8611,-1.8765,-3.5150,-0.8153,-0.8298,64.2350
2,-28.0000,0.0041,5.5019,0.1652,0.1652,0.1652,0.1652,0.1652,0.1652,0.1652,0.1652,0.1652,0.1652,-1.7892,-2.7892,1.9140,-1.6684,3.7032,-0.8577,-1.8729,-3.4576,-0.8140,-0.8285,67.3670
3,-27.0000,0.0041,5.5059,0.1665,0.1665,0.1665,0.1665,0.1665,0.1665,0.1665,0.1665,0.1665,0.1665,-1.7573,-2.7573,1.8747,-1.6429,3.6321,-0.8542,-1.8691,-3.4003,-0.8126,-0.8271,70.6200
4,-26.0000,0.0040,5.5100,0.1677,0.1677,0.1677,0.1677,0.1677,0.1677,0.1677,0.1677,0.1677,0.1677,-1.7256,-2.7256,1.8358,-1.6175,3.5614,-0.8506,-1.8652,-3.3431,-0.8110,-0.8255,73.9960


In [4]:
# KIZGIN FAZ ANALİZİ
# K-Fold, Performans Metrikleri, Öznitelik Katkıları ve Excel Raporlaması tek seferde çalıştırılır.

sup_metrics, sup_importances, sup_final_matrix = run_unified_analysis(
    df=df_kizgin,                  
    input_cols=SUPERHEATED_INPUTS, 
    outputs_list=SUPERHEATED_OUTPUTS, 
    dataset_name="Kizgin_Faz"
)

print("\n--- Eğitime Giren Nihai Zenginleştirilmiş Veri Matrisi (İlk 5 Satır) ---")
display(sup_final_matrix.head())


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  KIZGIN_FAZ - KESİN SIRALI K-FOLD ANALİZİ VE EXCEL RAPORLAMA
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

[FAZ 1/3] Her hedef değişken için ayrı ayrı hibrit veri setleri üretiliyor...
  -> Hedef (v (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (h (çıktı)) için STGP-EF çalıştırılıyor...
  -> Hedef (s (çıktı)) için STGP-EF çalıştırılıyor...

[FAZ 2/3] Orijinal (Base) veri setleri ile modeller eğitiliyor...
      -> AdaBoost eğitiliyor (Base)...
      -> CatBoost eğitiliyor (Base)...
      -> DART eğitiliyor (Base)...
      -> ET eğitiliyor (Base)...
      -> GBDT eğitiliyor (Base)...
      -> KNN eğitiliyor (Base)...
      -> LightGBM eğitiliyor (Base)...
      -> RF eğitiliyor (Base)...
      -> XGBoost eğitiliyor (Base)...
      -> AdaBoost eğitiliyor (Base)...
      -> CatBoost eğitiliyor (Base)...
      -> DART eğitiliyor (Base)...
      -> ET eğitiliyor (Base)...
      -> GBDT eğitiliyor (Base)...
 

,T (girdi),P (girdi),STGP_00,STGP_01,STGP_02,STGP_03,STGP_04,STGP_05,STGP_06,STGP_07,STGP_08,STGP_09,EF_00,EF_01,EF_02,EF_03,EF_04,EF_05,EF_06,EF_07,EF_08,EF_09,Target
0,-30.0000,50.0000,805.7149,-3689.1880,-537.9341,-541.2175,392.4410,-974.8780,5731.2225,-16440.8840,906.5747,-227.5049,-0.7717,-0.2133,-1.2133,1.2133,0.2087,1.4722,-2.4266,-1.1843,-2.2133,0.1357,0.3357
1,-25.0000,50.0000,824.9297,-3737.8526,-544.6906,-552.6367,397.3853,-1000.6577,5816.1952,-16856.4747,930.7619,-232.6944,-0.7717,-0.2133,-1.2133,1.2133,0.2084,1.4722,-2.4266,-1.1742,-2.2133,0.1357,0.3433
2,-20.0000,50.0000,844.1445,-3786.5173,-551.4471,-564.0560,402.3296,-1026.7737,5901.1679,-17277.2527,955.2668,-237.8838,-0.7717,-0.2133,-1.2133,1.2133,0.2080,1.4722,-2.4266,-1.1628,-2.2133,0.1357,0.3508
3,-15.0000,50.0000,863.3592,-3835.1819,-558.2036,-575.4753,407.2739,-1053.2258,5986.1406,-17703.2179,980.0892,-243.0733,-0.7717,-0.2133,-1.2133,1.2133,0.2076,1.4722,-2.4266,-1.1502,-2.2133,0.1357,0.3582
4,-10.0000,50.0000,882.5740,-3883.8465,-564.9600,-586.8946,412.2182,-1080.0140,6071.1133,-18134.3704,1005.2291,-248.2627,-0.7717,-0.2133,-1.2133,1.2133,0.2071,1.4722,-2.4266,-1.1366,-2.2133,0.1357,0.3656


In [6]:
# TERMODİNAMİK TUTARLILIK TESTİ (Sentetik Uzayda)
# Modellerin fiziksel yasalara (monotonluk) ne kadar uyduğu test edilir.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from Functions2 import enrich_input_space

# Temsili Olarak: Doymuş Fazda P(çıktı) tahmini için 11 modelin eğitimi
df_enr, enr_cols = enrich_input_space(df_doymus, SATURATED_INPUTS)
X_raw = df_enr[enr_cols].values
y_raw = df_doymus['P(çıktı)'].values

trained_pipelines = {}
for name, model in get_regressors().items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_raw, y_raw)
    trained_pipelines[name] = pipe

# -30°C ile 50°C arasında 0.1 adımla fizik yasalarına uyum (ihlal yüzdesi) denetlenir
grid_results = check_thermodynamics_on_synthetic_grid(
    models_dict=trained_pipelines,
    input_cols=SATURATED_INPUTS,
    min_T=-30, max_T=50, step=0.1
)

print("\n--- Modellerin Fiziksel Kuralları İhlal Yüzdeleri ---")
display(grid_results.sort_values(by='Ihlal_Yuzdesi_(%)'))


--- Modellerin Fiziksel Kuralları İhlal Yüzdeleri ---


,Algoritma,Sentetik_Nokta_Sayisi,Fiziksel_Ihlal_Sayisi,Ihlal_Yuzdesi_(%)
0,AdaBoost,800,0,0.0000
2,DART,800,0,0.0000
3,ET,800,0,0.0000
7,RF,800,0,0.0000
6,LightGBM,800,0,0.0000
8,XGBoost,800,0,0.0000
4,GBDT,800,5,0.6258
1,CatBoost,800,6,0.7509
5,KNN,800,11,1.3767
